# Chapter 15 — Bias-Variance Tradeoff
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Decompose prediction error into bias, variance, and irreducible noise
- Visualise underfitting and overfitting
- Use cross-validation to navigate the tradeoff

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — The Bias-Variance Decomposition

Expected MSE for a point $x_0$:
$$E[(y - \hat f(x_0))^2] = \text{Bias}^2 + \text{Variance} + \sigma^2_{\varepsilon}$$

- **Bias**: error from wrong assumptions (underfitting)
- **Variance**: sensitivity to training set fluctuations (overfitting)
- **Irreducible noise**: unavoidable noise in the data

In [ ]:
# True function
f_true = lambda x: np.sin(2 * np.pi * x)
sigma_noise = 0.3
x_test = np.linspace(0, 1, 200)

n_train = 30
n_experiments = 200
degrees = [1, 4, 15]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, degree in zip(axes, degrees):
    predictions = []
    for _ in range(n_experiments):
        x_train = rng.uniform(0, 1, n_train)
        y_train = f_true(x_train) + rng.normal(0, sigma_noise, n_train)
        model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1e-6))
        model.fit(x_train.reshape(-1,1), y_train)
        predictions.append(model.predict(x_test.reshape(-1,1)))
    preds = np.array(predictions)
    bias2 = (preds.mean(axis=0) - f_true(x_test))**2
    variance = preds.var(axis=0)
    for p in preds[:20]:
        ax.plot(x_test, p, alpha=0.1, color="blue")
    ax.plot(x_test, f_true(x_test), "r-", lw=2, label="True")
    ax.plot(x_test, preds.mean(axis=0), "k--", lw=2, label="Mean pred")
    ax.set_title(f"Degree={degree}
Bias²={bias2.mean():.3f}  Var={variance.mean():.3f}")
    ax.legend(fontsize=8)
plt.suptitle("Bias-Variance Tradeoff", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Deriving the Decomposition

Let $y = f(x) + \varepsilon$ where $E[\varepsilon]=0$, $\text{Var}[\varepsilon]=\sigma^2$.

$$E[(y - \hat f)^2] = E[f^2 + \hat f^2 + \varepsilon^2 - 2f\hat f - 2f\varepsilon + 2\hat f\varepsilon]$$

Since $\varepsilon \perp \hat f$, after algebra:
$$= (f - E[\hat f])^2 + E[(\hat f - E[\hat f])^2] + \sigma^2$$
$$= \text{Bias}^2[\hat f] + \text{Var}[\hat f] + \sigma^2$$

In [ ]:
# Quantify bias² and variance for polynomial degrees
results = []
for degree in range(1, 16):
    preds = []
    for _ in range(300):
        x_train = rng.uniform(0, 1, n_train)
        y_train = f_true(x_train) + rng.normal(0, sigma_noise, n_train)
        model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1e-6))
        model.fit(x_train.reshape(-1,1), y_train)
        preds.append(model.predict(x_test.reshape(-1,1)))
    preds = np.array(preds)
    b2 = ((preds.mean(0) - f_true(x_test))**2).mean()
    v  = preds.var(0).mean()
    results.append((degree, b2, v, b2+v+sigma_noise**2))

degs, b2s, vs, total = zip(*results)
plt.figure(figsize=(9,5))
plt.plot(degs, b2s, "b-o", label="Bias²")
plt.plot(degs, vs,  "r-o", label="Variance")
plt.plot(degs, total, "k-o", label="Total MSE")
plt.axhline(sigma_noise**2, ls="--", color="gray", label=f"Irreducible (σ²={sigma_noise**2:.2f})")
plt.xlabel("Polynomial Degree"); plt.ylabel("Error")
plt.title("Bias-Variance Tradeoff Curve")
plt.legend(); plt.tight_layout(); plt.show()

## 3. Simulation — Regularisation Controls the Tradeoff

**Ridge regression** adds $\alpha\|w\|^2$ to the loss, shrinking weights → increasing bias
but decreasing variance.

In [ ]:
alphas = np.logspace(-4, 4, 50)
cv_scores = []
for alpha in alphas:
    model = make_pipeline(PolynomialFeatures(10), Ridge(alpha=alpha))
    x_all = rng.uniform(0, 1, 150)
    y_all = f_true(x_all) + rng.normal(0, sigma_noise, 150)
    scores = cross_val_score(model, x_all.reshape(-1,1), y_all,
                             cv=5, scoring="neg_mean_squared_error")
    cv_scores.append(-scores.mean())

best_alpha = alphas[np.argmin(cv_scores)]
plt.semilogx(alphas, cv_scores)
plt.axvline(best_alpha, color="red", linestyle="--", label=f"Best α={best_alpha:.4f}")
plt.xlabel("Regularisation α"); plt.ylabel("CV MSE")
plt.title("Cross-Validation Chooses Optimal Regularisation")
plt.legend(); plt.tight_layout(); plt.show()

## 4–6. Visualisation, Real Dataset & Practice

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X, y = housing.data, housing.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

alphas_lasso = np.logspace(-3, 1, 30)
train_errs, cv_errs = [], []
for a in alphas_lasso:
    m = Lasso(alpha=a, max_iter=5000)
    train_score = cross_val_score(m, X_scaled, y, cv=5, scoring="neg_mean_squared_error")
    cv_errs.append(-train_score.mean())

plt.semilogx(alphas_lasso, cv_errs, "b-o")
plt.xlabel("Lasso α"); plt.ylabel("CV MSE")
plt.title("California Housing — Lasso Regularisation Path")
plt.tight_layout(); plt.show()

In [ ]:
# Practice P1: compute bias and variance for kNN with k=1,5,20
from sklearn.neighbors import KNeighborsRegressor

x_tr = rng.uniform(0, 1, 50)
y_tr = f_true(x_tr) + rng.normal(0, sigma_noise, 50)

for k in [1, 5, 20]:
    preds_k = []
    for _ in range(200):
        xi = rng.uniform(0, 1, 50)
        yi = f_true(xi) + rng.normal(0, sigma_noise, 50)
        m = KNeighborsRegressor(n_neighbors=k)
        m.fit(xi.reshape(-1,1), yi)
        preds_k.append(m.predict(x_test.reshape(-1,1)))
    preds_k = np.array(preds_k)
    b2 = ((preds_k.mean(0) - f_true(x_test))**2).mean()
    v  = preds_k.var(0).mean()
    print(f"k={k:>2}: Bias²={b2:.4f}, Var={v:.4f}, Total≈{b2+v+sigma_noise**2:.4f}")